<img src="https://github.com/v-iashin/v-iashin.github.io/raw/master/images/bmt/bi_modal_transformer.svg" alt="Bi-Modal Transformer with Proposal Generator" width="900">


# Dense Video Captioning with Bi-modal Transformer
[Project Page](https://v-iashin.github.io/bmt) • [ArXiv](https://arxiv.org/abs/2005.08271) • [BMVC Page](https://www.bmvc2020-conference.com/conference/papers/paper_0111.html) • [Presentation](https://www.youtube.com/watch?v=C4zYVIqGDVQ)

This notebook accompanies the [source code](https://github.com/v-iashin/BMT) of the paper:
_A Better Use of Audio-Visual Cues: Dense Video Captioning with Bi-modal Transformer_ (BMVC 2020).

Use **Google Colab with an NVIDIA GPU** for the full pipeline (feature extraction + captioning), **or macOS with Apple Silicon** for captioning on **PyTorch MPS (Metal)** once I3D/VGGish `.npy` features exist (extract them on Colab/Linux—the upstream extractors need CUDA).

A first-time Colab run still takes on the order of ~30 minutes including checkpoints and conda environments.



## Cloning the Repository

In [1]:
#@title
# Clone BMT (skip if you already have the repo next to this notebook or open the notebook from inside `BMT/`)
import os
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "conda_env.yml").is_file() or (cwd / "conda_env_mac_metal.yml").is_file():
    print("Already in BMT repo:", cwd)
elif (cwd / "BMT" / "conda_env.yml").is_file():
    os.chdir(cwd / "BMT")
    print("Entered", Path.cwd())
else:
    if (cwd / "BMT").exists():
        raise RuntimeError(
            "Directory ./BMT exists but does not look like the BMT repo. Remove or rename it, then re-run."
        )
    import subprocess
    subprocess.run(
        ["git", "clone", "--recursive", "https://github.com/v-iashin/BMT.git"],
        check=True,
    )
    os.chdir(cwd / "BMT")
    print("Cloned and entered", Path.cwd())


Cloning into 'BMT'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 182 (delta 42), reused 40 (delta 37), pack-reused 124 (from 1)
Receiving objects: 100% (182/182), 12.86 MiB | 11.86 MiB/s, done.
Resolving deltas: 100% (77/77), done.
Submodule 'submodules/pycocoevalcap' (https://github.com/salaniz/pycocoevalcap.git) registered for path 'submodules/pycocoevalcap'
Submodule 'submodules/video_features' (https://github.com/v-iashin/video_features.git) registered for path 'submodules/video_features'
Cloning into '/Users/numaan/Cost-Evaluation-Portal-v3/BMT/submodules/pycocoevalcap'...
remote: Enumerating objects: 821, done.        
remote: Counting objects: 100% (12/12), done.        
remote: Compressing objects: 100% (9/9), done.        /9)        
remote: Total 821 (delta 4), reused 3 (delta 3), pack-reused 809 (from 2)        
Receiving objects: 100% (821/821), 130.06 MiB | 9.95 MiB/s,

## Installing Environments
Since feature extraction and the captioning model rely on different Python environments, the original authors use `conda`.

- **Google Colab (Linux)**: install Miniconda under `/usr/local` and use the original CUDA `conda_env*.yml` files.
- **macOS (Apple Silicon)**: use your existing Miniconda/Anaconda, create the `bmt_mps` env from `conda_env_mac_metal.yml` (PyTorch with **MPS**), and bring your own pre-extracted `.npy` features from a CUDA machine or Colab.


**Colab (Linux):** patch encoding / SSL, install Miniconda, then create conda envs.

**macOS:** skip the next two cells (or run them only on Colab); start from the runtime paths cell after you `conda env create -f conda_env_mac_metal.yml` inside the cloned `BMT` repo.


In [ ]:
#@title
# Colab-only: SSL / encoding tweaks (skipped on macOS / local)
import locale
import os

IN_COLAB = os.path.isdir("/content")
if IN_COLAB:
    locale.getpreferredencoding = lambda: "UTF-8"
    !pip install cryptography==38.0.4
    !pip install pyopenssl --upgrade
else:
    print("Skipping Colab SSL pins (not in /content).")


  Obtaining dependency information for cryptography==38.0.4 from https://files.pythonhosted.org/packages/75/7a/2ea7dd2202638cf1053aaa8fbbaddded0b78c78832b3d03cafa0416a6c84/cryptography-38.0.4-cp36-abi3-macosx_10_10_universal2.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━ 4.2/5.4 MB 16.9 MB/s eta 0:00:01

In [ ]:
#@title
# Miniconda for Google Colab (Linux x86_64 only)
import os
import shutil
import sys

IN_COLAB = os.path.isdir("/content")
if not IN_COLAB or not sys.platform.startswith("linux"):
    print("Skipping Miniconda installer (use on Google Colab Linux, or use your own conda on macOS).")
else:
    INSTALLER = "Miniconda3-py37_23.1.0-1-Linux-x86_64.sh"
    URL = "https://repo.anaconda.com/miniconda/" + INSTALLER
    if not os.path.exists(INSTALLER):
        if shutil.which("wget"):
            get_ipython().system(f"wget {URL} -q --show-progress -O {INSTALLER}")
        elif shutil.which("curl"):
            get_ipython().system(f"curl -L {URL} -o {INSTALLER}")
        else:
            raise RuntimeError("Need wget or curl.")
    get_ipython().system(f"bash ./{INSTALLER} -b -f -p /usr/local")


Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: pyopenssl
    Found existing installation: pyOpenSSL 22.0.0
    Uninstalling pyOpenSSL-22.0.0:
      Successfully uninstalled pyOpenSSL-22.0.0


In [ ]:
#@title
# Runtime paths: Colab vs macOS (import shared helpers from `BMT/bmt_colab_helpers.py`)
import sys
from pathlib import Path

_helpers_dir = None
for _d in [Path.cwd(), Path.cwd() / "BMT", *list(Path.cwd().parents)[:8]]:
    if (_d / "bmt_colab_helpers.py").is_file():
        _helpers_dir = _d
        break
if _helpers_dir is None:
    raise FileNotFoundError(
        "bmt_colab_helpers.py not found. Add it to your BMT clone root (see Cost-Evaluation-Portal-v3/BMT)."
    )
sys.path.insert(0, str(_helpers_dir))

import bmt_colab_helpers as bch

rt = bch.setup_bmt_notebook()
IN_COLAB = rt.in_colab
ON_MAC = rt.on_mac
CONDA_ROOT = rt.conda_root
BMT_ROOT = rt.bmt_root
BMT_PY = rt.bmt_python
I3D_PY = rt.i3d_python
VGGISH_PY = rt.vggish_python
get_video_duration = bch.get_video_duration


In [ ]:
#@title
# Create conda environments (Linux CUDA vs macOS MPS)
import os
import platform

IN_COLAB = os.path.isdir("/content")
ON_MAC = platform.system() == "Darwin"

if ON_MAC:
    # macOS (Apple Silicon): captioning on MPS only — use Colab/Linux for I3D+VGGish .npy
    print("Creating bmt_mps (PyTorch + MPS) for captioning …")
    !conda env create -f ./conda_env_mac_metal.yml
    !{BMT_PY} -m spacy download en_core_web_sm
    print("Skipping i3d / vggish conda envs on macOS (CUDA-only upstream stack).")
else:
    # feature extraction (i3d, vggish) + captioning model (bmt) — same as original Colab flow
    !export PIP_DEFAULT_TIMEOUT=100
    !conda env create -f ./submodules/video_features/conda_env_i3d.yml
    !conda env create -f ./submodules/video_features/conda_env_vggish.yml
    !conda env create -f ./conda_env.yml
    # spacy language model (spaCy 2.x expects `en`; macOS bmt_mps uses en_core_web_sm above)
    !{BMT_PY} -m spacy download en



Streaming output truncated to the last 5000 lines.






cudnn-7.3.1          | 286.1 MB  | :  59% 0.5947749567735988/1 [00:17<01:19, 196.95s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5953211730990406/1 [00:17<01:20, 199.35s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5958673894244825/1 [00:17<01:21, 201.11s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5964136057499243/1 [00:17<01:22, 203.83s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5969598220753661/1 [00:17<01:22, 204.99s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5974514167682637/1 [00:17<01:23, 206.25s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5979430114611614/1 [00:18<01:22, 205.93s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5984346061540591/1 [00:18<01:23, 208.23s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5989262008469567/1 [00:18<01:25, 212.14s/it]









cudnn-7.3.1          | 286.1 MB  | :  60% 0.5994724171723985/

## Downloading Checkpoints

Downloading the GloVe embeddings. `torchtext` could do it for us but it uses a slow server.
Therefore, we store it on our premises.

In [ ]:
#@title
# Download checkpoints (wget or curl)
import shutil
import os

def fetch(url, out):
    if shutil.which("wget"):
        get_ipython().system(f"wget {url} -q --show-progress -O {out}")
    else:
        get_ipython().system(f"curl -L {url} -o {out}")

fetch("https://a3s.fi/swift/v1/AUTH_a235c0f452d648828f745589cde1219a/bmt/glove.840B.300d.zip", "glove.840B.300d.zip")
fetch("https://a3s.fi/swift/v1/AUTH_a235c0f452d648828f745589cde1219a/bmt/best_cap_model.pt", "best_cap_model.pt")
fetch("https://a3s.fi/swift/v1/AUTH_a235c0f452d648828f745589cde1219a/bmt/best_prop_model.pt", "best_prop_model.pt")
fetch("https://storage.googleapis.com/audioset/vggish_model.ckpt", "vggish_model.ckpt")

!mkdir -p .vector_cache sample submodules/video_features/models/vggish/checkpoints
!mv glove.840B.300d.zip ./.vector_cache/
!mv best_cap_model.pt ./sample/
!mv best_prop_model.pt ./sample/
!mv vggish_model.ckpt ./submodules/video_features/models/vggish/checkpoints/



glove.840B.300d.zip 100%[===================>]   2.03G  66.0MB/s    in 31s     
best_cap_model.pt   100%[===================>] 589.61M  64.1MB/s    in 9.4s    
best_prop_model.pt  100%[===================>]   2.50G  15.4MB/s    in 1m 50s  
vggish_model.ckpt   100%[===================>] 277.62M  28.7MB/s    in 10s     


## Running Inference on a Custom Video

Upload your video (or use the bundled sample). Set `MY_VIDEO_PATH` below.

On **macOS**, point to a local path; feature `.npy` files must already exist (from Colab/Linux extraction).


In [ ]:
#@title
# Video + checkpoint paths (bundled `sample/*.npy` when present)
import os
from pathlib import Path

import bmt_colab_helpers as bch

MY_VIDEO_PATH = str(BMT_ROOT / "sample/women_long_jump.mp4")

VIDEO_DURATION = get_video_duration(MY_VIDEO_PATH)

feat = bch.resolve_feature_npy_paths(BMT_ROOT, MY_VIDEO_PATH)
FEATURES_CACHE_PATH = str(feat.cache_dir) + os.sep
os.makedirs(feat.cache_dir, exist_ok=True)
FEATURE_PATH_RGB = str(feat.rgb)
FEATURE_PATH_FLOW = str(feat.flow)
FEATURE_PATH_VGGISH = str(feat.vggish)

PROPOSAL_CKPT = str(BMT_ROOT / "sample/best_prop_model.pt")
CAPTIONING_CKPT = str(BMT_ROOT / "sample/best_cap_model.pt")

print("Feature npy paths:")
print(" ", FEATURE_PATH_RGB)
print(" ", FEATURE_PATH_FLOW)
print(" ", FEATURE_PATH_VGGISH)


Video Duration: 35.155


The script will extract audio and visual features from the video

In [ ]:
#@title
# Feature extraction (CUDA / Colab only)
import os
import platform
import subprocess

ON_MAC = platform.system() == "Darwin"

if ON_MAC:
    print(
        "Skipping I3D/VGGish extraction on macOS (requires CUDA + CuPy + TF1 in upstream video_features).\n"
        "If you do not yet have feature files, run this notebook on Colab through the extraction cells, "
        f"then copy:\n  {FEATURE_PATH_RGB}\n  {FEATURE_PATH_FLOW}\n  {FEATURE_PATH_VGGISH}\n"
        "to this machine (same names under tmp/)."
    )
else:
    i3d_cwd = os.path.join(os.getcwd(), "submodules", "video_features")
    subprocess.run(
        [
            I3D_PY,
            "main.py",
            "--feature_type",
            "i3d",
            "--on_extraction",
            "save_numpy",
            "--device_ids",
            "0",
            "--extraction_fps",
            "25",
            "--video_paths",
            MY_VIDEO_PATH,
            "--output_path",
            FEATURES_CACHE_PATH,
        ],
        cwd=i3d_cwd,
        check=True,
    )
    subprocess.run(
        [
            VGGISH_PY,
            "main.py",
            "--feature_type",
            "vggish",
            "--on_extraction",
            "save_numpy",
            "--device_ids",
            "0",
            "--video_paths",
            MY_VIDEO_PATH,
            "--output_path",
            FEATURES_CACHE_PATH,
        ],
        cwd=i3d_cwd,
        check=True,
    )



Saving features to /content/BMT/tmp/
100% 1/1 [00:48<00:00, 48.85s/it]
Saving features to /content/BMT/tmp/

For more information, please see:
  * https://github.com/tensorflow/community/blob/master/rfcs/20180907-contrib-sunset.md
  * https://github.com/tensorflow/addons
If you depend on functionality not listed there, please file an issue.

100% 1/1 [00:12<00:00, 12.54s/it]


Run dense video captioning

In [ ]:
#@title
# Dense video captioning (uses CUDA on Colab; on Apple Silicon prefers MPS via --device auto)
import subprocess

MAX_PROP_PER_VIDEO = 100
NMS_TIOU_THRESHOLD = 0.4

subprocess.run(
    [
        BMT_PY,
        "./sample/single_video_prediction.py",
        "--prop_generator_model_path",
        PROPOSAL_CKPT,
        "--pretrained_cap_model_path",
        CAPTIONING_CKPT,
        "--vggish_features_path",
        FEATURE_PATH_VGGISH,
        "--rgb_features_path",
        FEATURE_PATH_RGB,
        "--flow_features_path",
        FEATURE_PATH_FLOW,
        "--duration_in_secs",
        str(VIDEO_DURATION),
        "--device",
        "auto",
        "--max_prop_per_vid",
        str(MAX_PROP_PER_VIDEO),
        "--nms_tiou_thresh",
        str(NMS_TIOU_THRESHOLD),
    ],
    cwd=str(BMT_ROOT),
    check=True,
)



Contructing caption_iterator for "train" phase
100% 2196016/2196017 [04:12<00:00, 8691.19it/s] 
Using vanilla Generator
initialization: xavier
Glove emb of the same size as d_model_caps
Pretrained caption path: 
 /content/BMT/sample/best_cap_model.pt
[{'start': 0.1, 'end': 4.9, 'sentence': 'We see a title screen'}, {'start': 5.0, 'end': 7.9, 'sentence': 'A large group of people are seen standing around a building'}, {'start': 0.7, 'end': 11.9, 'sentence': 'A man is seen standing in front of a large crowd'}, {'start': 19.6, 'end': 33.3, 'sentence': 'The woman runs down a track and jumps into a sand pit'}, {'start': 7.5, 'end': 10.0, 'sentence': 'A large group of people are seen standing around a building'}, {'start': 0.6, 'end': 35.1, 'sentence': 'A large group of people are seen running down a track while others watch on the sides'}, {'start': 8.2, 'end': 13.7, 'sentence': 'A man runs down a track'}, {'start': 0.1, 'end': 2.0, 'sentence': 'We see a title screen'}]
